### Pinecone Vector DB

Create your index and apikey from here https://www.pinecone.io/

In [1]:
## Import necessary libraries
import os
from dotenv import load_dotenv

load_dotenv()

## Get the api key
api_key = os.getenv('PINECONE_API_KEY')

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone

## Get the pinecone
pc = Pinecone(api_key = api_key)

## get the embeddings model
## Initialize a simple HuggingFace embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5"
)

embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Udemy Material\4-Retrieval-Augmented-Generation\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dhruv\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

HuggingFaceEmbeddings(model_name='BAAI/bge-large-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [3]:
from pinecone import ServerlessSpec
index_name = "ragtest"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 1024,
        metric = "cosine",
        spec = ServerlessSpec(cloud = "aws", region = "us-east-1"),
    )

index = pc.Index(index_name)
index

In [4]:
from langchain_pinecone import PineconeVectorStore

## Get the vector store for pinecone
vector_store = PineconeVectorStore(
    index_name = index_name,
    embedding = embeddings,
    pinecone_api_key = api_key
)

vector_store

In [5]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [6]:
## Add the documents to the vector store
vector_store.add_documents(documents=documents)

['c28c2b1c-21d0-4e69-afc3-0eecd10be199',
 'b283a1b1-8fec-4502-9d40-31d9081fa3c8',
 'db4d07cf-6013-42f5-81b1-19aab52fdeee',
 'e26372f0-781e-4689-b2c6-464343011ff8',
 'e68a0cfc-31ba-4095-b36d-572ddcac991f',
 'c7b9ae93-0549-46fc-b5b0-a746f4a5127f',
 'c85d3cb8-c34f-4502-97d6-22849e5c10af',
 '12b1ead9-4d8b-40b4-ab39-50c9952f0700',
 'd74a9d05-ab00-496c-bf69-a2d771641637',
 'ac941321-ceb6-4d07-be06-39ca0fc87791']

In [7]:
## Query Directly
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [8]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.672889] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [9]:
## Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)

retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='e26372f0-781e-4689-b2c6-464343011ff8', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]